## Task 0 — Setup

If a package is missing, install it *from inside the notebook* with `%pip install`
(the `%` magic targets the kernel's interpreter — a bare `!pip install` can install into
a different Python than the one running your kernel, which is a classic hour-long
debugging detour).

In [1]:
# How to download missing packages before import in the Jupyter notebook

%pip install pandas pyarrow ipykernel


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
## Notebook display settings: wide enough to see nested column names, but not so many
# rows that a stray `df` floods the output.

import json
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)

traderspost = Path("traderspost_signals_1786583992.csv")
if not traderspost.exists():
    traderspost = Path("/Users/efiles/dev/mym_algo/docs/data/traderspost_signals_1786583992.csv")

print(pd.__version__ , "|" , traderspost.resolve())



3.0.1 | /Users/efiles/dev/mym_algo/docs/data/traderspost_signals_1786583992.csv


### Error Codes from execution process

Error: 
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
/var/folders/y0/dwt2yhxj3l74m46b57v8b5v00000gn/T/ipykernel_19194/8231998.py in ?()
      5 pd.set_option("display.width", 200)
      6 pd.set_option("display.max_rows", 60)
      7 
      8 traderpost = pd.read_csv(r'/Users/efiles/Downloads/traderspost_signals_1786583992.csv')
----> 9 if not traderpost.exists():
     10     traderpost = Path("/Users/efiles/dev/mym_algo/docs/data/traderspost_signals_1786583992.csv")
     11 #####
     12 

~/prod/em_algo/env/lib/python3.11/site-packages/pandas/core/generic.py in ?(self, name)
   6202             and name not in self._accessors
   6203             and self._info_axis._can_hold_identifiers_and_holds_name(name)
   6204         ):
   6205             return self[name]
-> 6206         return object.__getattribute__(self, name)

AttributeError: 'DataFrame' object has no attribute 'exists'


My Code:
## Notebook display settings: wide enough to see nested column names, but not so many
# rows that a stray `df` floods the output.

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)

traderpost = pd.read_csv(r'/Users/efiles/Downloads/traderspost_signals_1786583992.csv')
if not traderpost.exists():
    traderpost = Path("/Users/efiles/dev/mym_algo/docs/data/traderspost_signals_1786583992.csv")
#####

#----
#
#
#
#####
print(pd.__version__ , "|" , traderspost.resolve())

# TraderPost Data Wrangler Commands copied from the transformations (thru pandas)

#import pandas 
import pandas as p

#load data as raw data
traderpost = p.read_csv(r'/Users/efiles/Downloads/traderspost_signals_1786583992 (1).csv')

# never transform the raw file
df = traderpost

df.head(50)

In [ ]:
## Helper Function for Automated Ticker Selection

from datetime import date

def get_front_month_symbol(base="MYM"):
    """
    Returns the current MYM front-month futures symbol.
    Rolls: March (H), June (M), September (U), December (Z)
    """
    today  = date.today()
    month  = today.month
    year   = str(today.year)[-1]  # last digit of year

    if month <= 3:   suffix = f"H{year}"
    elif month <= 6: suffix = f"M{year}"
    elif month <= 9: suffix = f"U{year}"
    else:            suffix = f"Z{year}"

    return f"{base}{suffix}"

print(get_front_month_symbol())  # e.g. MYMH6 in Q1 2026

## Task 1 — Load and profile before you touch anything

**Why:** the first question is never "how do I reshape this" but "what is actually in
here". A 30-second profile tells you which columns are dead weight, which are keys, and
which need parsing.

**Common approaches:** `df.info()` for dtypes and non-null counts, `df.describe(include="all")`
for distributions, and a hand-rolled profile DataFrame when you want exactly the columns
you care about. For a heavier lift, `ydata-profiling` generates a full HTML report — but
on a 9-column file it's slower to read than to write the four lines below.

**Pitfall:** pandas type inference on a mixed file is a guess. `Message Code` is empty in
every row, so pandas infers `float64` and fills it with `NaN` — an entirely fictional
dtype for a column that was never numeric. Loading with `dtype=str` keeps every value
exactly as the file has it and lets *you* decide the types in Task 7.

**Your turn:**
1. Load the CSV into `raw` with **every column as string** (`dtype=str`).
2. Build a `report` DataFrame indexed by column name, with columns
   `dtype`, `n_missing`, `n_unique` (count `NaN` as a value), and `sample` (row 0).
3. Store in `dead_cols` the column names that carry **no information** — one distinct
   value or fewer across the whole file.

In [4]:
raw = pd.read_csv(traderspost, dtype=str)

data_report =  pd.DataFrame({
"d_type": raw.dtypes.astype(str),
"n_missing": raw.isna().sum(),
"n_unique": raw.nunique(dropna=False),    
"sample": raw.iloc[0],    
})

# nunique(dropna=False) counts NaN as a value, so an all-empty column scores 1, not 0.
dead_cols = data_report.index[data_report["n_unique"] <= 1].tolist()

print(raw.shape)
data_report

(435, 9)


,d_type,n_missing,n_unique,sample
UUID,str,0,435,938c103b-ed5e-4467-a43a-255945fd7336
Created At,str,0,435,2026-08-11 18:03:37.148839
Payload,str,0,76,"{""test"": false, ""ticker"": ""MYM"", ""action"": ""bu..."
Method,str,0,1,POST
URL,str,0,1,https://webhooks.traderspost.io/trading/webhoo...
User Agent,str,0,1,python-requests/2.32.5
IP Address,str,0,17,74.220.49.49
Message Code,str,435,1,NaN
Status Code,str,0,1,200


In [5]:
# --- Check ---
assert raw.shape == (435, 9), f"expected 435x9, got {raw.shape}"
assert (raw.dtypes == object).all() or all(str(t) in {"object", "str"} for t in raw.dtypes), \
    "load with dtype=str so nothing is silently coerced"
assert set(dead_cols) == {"Method", "URL", "User Agent", "Message Code", "Status Code"}, dead_cols
print("✅ Task 1 — every request is a 200 POST from the same script to the same webhook URL.")
print("   All the variation lives in Payload, Created At, and IP Address.")

✅ Task 1 — every request is a 200 POST from the same script to the same webhook URL.
   All the variation lives in Payload, Created At, and IP Address.


## Task 2 — Prune the dead columns and normalize the names

**Why:** `raw["Created At"]` needs brackets and exact spacing; `df.created_at` doesn't.
Snake-casing your columns immediately means every downstream cell can use attribute
access and `query()` strings, and it removes a whole class of typo bugs.

**Common approaches:** `df.rename(columns=str.lower)` for the trivial case, a `re.sub`
one-liner for spaces and punctuation, or `df.columns = df.columns.str.replace(...)` with
the `.str` accessor. Libraries like `pyjanitor` ship a `clean_names()` that does this and
more, but a five-line function you can read beats a dependency you can't.

The converter below also splits **camelCase**, which you don't need yet — the CSV headers
are `Title Case` — but you will in Task 6, when the JSON keys arrive as `signalPrice` and
`stopLoss`. Write it once, use it twice.

**Pitfall:** "constant column" and "useless column" aren't synonyms. `URL` is identical in
all 435 rows *in this export*, but it encodes which webhook endpoint fired — meaningful the
moment you concatenate a second strategy's export. Dropping it here is fine because the
Bonus task pulls the endpoint ID from `raw`, which you never modify.

**Your turn:**
1. Finish `to_snake` so it lowercases, splits camelCase, replaces runs of non-word
   characters with `_`, collapses repeats, and strips leading/trailing underscores.
2. Build `df` = `raw` minus `dead_cols`, with snake_cased column names.

In [6]:
# Define a data cleaning function for the payload data that is named "to_snake"
## the function will take a string as an argument and output the cleaned string based on the functions given

def to_snake(name: str) -> str:
    """'Created At' -> 'created_at', 'signalPrice' -> 'signal_price', 'untouched?' -> 'untouched'"""
    name = name.replace("extras__", "")                    # used in Task 6
    name = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)    # camelCase -> camel_Case
    name = re.sub(r"\W+", "_", name)                       # spaces, ?, - -> _
    name = re.sub(r"_+", "_", name)                        # collapse runs
    return name.strip("_").lower()

df = raw.drop(columns=dead_cols).rename(columns=to_snake)

df.head(3)

,uuid,created_at,payload,ip_address
0,938c103b-ed5e-4467-a43a-255945fd7336,2026-08-11 18:03:37.148839,"{""test"": false, ""ticker"": ""MYM"", ""action"": ""bu...",74.220.49.49
1,e8289c0c-923d-4421-a536-febfeb64a5b3,2026-08-11 17:57:01.246374,"{""test"": false, ""ticker"": ""MYM"", ""action"": ""bu...",74.220.49.49
2,5edb7ecb-5eb0-4c46-a095-c1bda8cf1193,2026-08-11 17:43:12.358124,"{""test"": false, ""ticker"": ""MYM"", ""action"": ""bu...",74.220.49.49


In [7]:
# --- Check ---
assert list(df.columns) == ["uuid", "created_at", "payload", "ip_address"], list(df.columns)
assert to_snake("stopLoss") == "stop_loss" and to_snake("untouched?") == "untouched"
assert to_snake("extras__potential_t1") == "potential_t1"
assert raw.shape == (435, 9), "don't mutate raw — derive df from it"
print("✅ Task 2 — 9 columns down to 4, and to_snake is ready for the JSON keys.")

✅ Task 2 — 9 columns down to 4, and to_snake is ready for the JSON keys.


## Task 3 — Timestamps and time zones

**Why:** `"2026-08-11 18:03:37.148839"` is a string. Sorting it works by luck (ISO-8601
sorts lexicographically); everything else — differences, resampling, `.dt.hour` — does not.

**Common approaches:** `pd.to_datetime(series)` is the standard move. Pass an explicit
`format=` when you know it — it's dramatically faster than inference on large files and it
*fails loudly* on rows that don't match, which is what you want. Use `format="mixed"` when
formats genuinely vary, and `errors="coerce"` to turn unparseable values into `NaT`
instead of an exception (then count the `NaT`s — silent coercion is how bad rows sneak
through).

**Pitfall — the big one:** these timestamps are **naive**, meaning they carry no zone. The
TradersPost log records UTC. Payload timestamps you'll parse in Task 8 carry an explicit
`-04:00` offset. Subtracting a naive timestamp from an aware one raises `TypeError`, and
the fix people reach for — stripping the offset — silently introduces a 4-hour error into
every duration. Rule: **localize on the way in, convert only for display.**

**Your turn:**
1. Parse `created_at` to datetime and mark it as UTC with `.dt.tz_localize("UTC")`.
2. Add `created_at_et`, the same instants converted to `America/New_York` (market time).
3. Add `session_date` — the ET calendar date — as a real `date`, not a string.

In [8]:
# tz_localize attaches a zone to naive stamps; tz_convert moves an aware stamp to another.
df["created_at"] = pd.to_datetime(df["created_at"], format="mixed").dt.tz_localize("UTC")
df["created_at_et"] = df["created_at"].dt.tz_convert("America/New_York")
df["session_date"] = df["created_at_et"].dt.date

df[["created_at", "created_at_et", "session_date"]].head(3)

,created_at,created_at_et,session_date
0,2026-08-11 18:03:37.148839+00:00,2026-08-11 14:03:37.148839-04:00,2026-08-11
1,2026-08-11 17:57:01.246374+00:00,2026-08-11 13:57:01.246374-04:00,2026-08-11
2,2026-08-11 17:43:12.358124+00:00,2026-08-11 13:43:12.358124-04:00,2026-08-11


In [9]:
# --- Check ---
assert "UTC" in str(df["created_at"].dtype), df["created_at"].dtype
assert "New_York" in str(df["created_at_et"].dtype), df["created_at_et"].dtype
assert df["created_at"].notna().all(), "no NaT allowed — every row parsed"
assert df["created_at"].max().tz_convert("America/New_York").hour == 14
print("✅ Task 3 — the last webhook fired 2026-08-11 14:03 ET (18:03 UTC).")
print("   Note both columns describe the SAME instants; only the display zone differs.")

✅ Task 3 — the last webhook fired 2026-08-11 14:03 ET (18:03 UTC).
   Note both columns describe the SAME instants; only the display zone differs.


Snake case is a naming convention where you write a multi-word name all lowercase, with underscores (_) separating the words — like a snake slithering between them.

"Renaming columns to make them table ready"

Examples:
"Created At" → created_at
"stopLoss" (camelCase) → stop_loss
"User Agent" → user_agent

## Task 4 — JSON strings → Python objects, defensively

**Why:** everything interesting is a string that *looks* like a dict. Until you parse it,
`payload.str.contains("buy")` is the best you can do — and it will happily match the word
"buy" inside an unrelated field.

**Common approaches:** `df["payload"].map(json.loads)` is the idiom. `.apply()` does the
same thing for a Series; `.map()` is marginally faster and signals element-wise intent.
Wrap `json.loads` when the source is untrusted. (Reflex to unlearn: `ast.literal_eval` is
for Python literals — it chokes on JSON's `true`/`false`/`null`. Use it only for logs that
contain `repr()`-ed dicts.)

**Pitfall:** one malformed blob in row 300 aborts the whole `.map()` and tells you nothing
about which row broke. A `safe_loads` that returns `None` on failure lets you parse
everything, then count and inspect the failures — the difference between a five-second
loop and a twenty-minute bisect. This file happens to be clean, which you can only know
*after* you check.

**Your turn:**
1. Write `safe_loads(s)` → parsed object, or `None` on `JSONDecodeError`/`TypeError`.
2. Apply it to `df["payload"]` → `parsed` (a Series of dicts).
3. Count failures in `n_bad`.
4. Build `shape_counts`: a `Counter` of each distinct **top-level key set**
   (`tuple(sorted(d))`) → how many rows have it.

"Idiom" here means: the conventional, expected way experienced practitioners write this in a given language/library — not the only way that works, but the way other people writing that language will recognize instantly, and the way the language's own style favors.

Where it showed up in your notebook, Task 4:

df["payload"].map(json.loads) is the idiom.

Meaning: yes, you could write a for loop, or a list comprehension, or .apply() — they'd all produce the same result. But .map() applied element-wise over a Series is what a pandas-fluent person reaches for by reflex, so it's "the idiom" — the default, recognizable, expected pattern for "apply this function to every value in this column."

It's a "best practices" word: writing idiomatic code means writing in the grain of the language/library rather than against it, which usually also means it's more readable, sometimes faster (idioms are often optimized internally), and easier for others (or future-you) to maintain.

In [8]:
from collections import Counter

def safe_loads(s):
    try:
        return json.loads(s)
    except (json.JSONDecodeError, TypeError):
        return None   # keep the row, flag it, inspect later

parsed = df["payload"].map(safe_loads)
n_bad = parsed.isna().sum()
shape_counts = Counter(tuple(sorted(d)) for d in parsed if d is not None)

print(f"{n_bad} rows failed to parse")
for keys, n in shape_counts.most_common():
    print(f"{n:>4}  {keys}")

0 rows failed to parse
 428  ('action', 'extras', 'orderType', 'price', 'quantity', 'signalPrice', 'stopLoss', 'takeProfit', 'test', 'ticker')
   2  ('action', 'cancel', 'ignoreTradingWindows', 'ticker')
   2  ('action', 'quantity', 'test', 'ticker')
   2  ('action', 'orderType', 'test', 'ticker')
   1  ('action', 'cancel', 'cancelOrderType', 'orderType', 'test', 'ticker')


In [9]:
# --- Check ---
assert n_bad == 0, f"{n_bad} unparseable rows"
assert parsed.map(type).eq(dict).all(), "every payload should be a dict"
assert len(shape_counts) == 5, f"expected 5 distinct key sets, got {len(shape_counts)}"
assert shape_counts.most_common(1)[0][1] == 428
print("✅ Task 4 — 428 rows share the full entry-signal schema; 7 rows are cancels,")
print("   exits and breakevens with 4-5 keys each. That ragged 7 drives Tasks 5-6.")

✅ Task 4 — 428 rows share the full entry-signal schema; 7 rows are cancels,
   exits and breakevens with 4-5 keys each. That ragged 7 drives Tasks 5-6.


## Task 5 — Flatten the nested JSON

**Why:** `parsed` is a Series of dicts — Python objects sitting in a pandas box. No
vectorized operation reaches inside them. You need one column per leaf value.

**Common approaches, worst to best:**

| Approach | Verdict |
|---|---|
| `df["payload"].str.extract(regex)` | Never. Regex on JSON is a bug generator. |
| `for _, row in df.iterrows(): ...` | Works, ~100× slower, tempting, avoid. |
| `parsed.apply(pd.Series)` | Fine for **flat** dicts. Leaves nested dicts as objects. |
| `pd.DataFrame(parsed.tolist())` | Same as above, a bit faster. |
| **`pd.json_normalize(parsed)`** | **The standard answer.** Recurses into nested dicts. |

`json_normalize` walks the tree and names leaves by their path: `{"stopLoss": {"amount": 31}}`
becomes column `stopLoss.amount`. Useful arguments: `sep=` (path separator), `max_level=`
(stop recursing at depth N and leave deeper dicts as objects), and `record_path`/`meta`
(for payloads containing *lists* of records — not needed here, essential the day a payload
has `"legs": [...]`).

**Pitfall:** the default `sep="."` produces `stopLoss.amount`, which breaks
`df.query("stopLoss.amount > 30")` and attribute access, because `.` means something else
in both. Use `sep="__"` and convert to snake_case in Task 6.

**Your turn:**
1. Build `wide = pd.json_normalize(parsed, sep="__")`.
2. Build `naive = parsed.apply(pd.Series)` and compare their column counts — that
   difference is exactly the nesting `apply(pd.Series)` failed to open up.

In [10]:
wide = pd.json_normalize(parsed, sep="__")
naive = parsed.apply(pd.Series)

print(f"json_normalize: {wide.shape[1]} columns")
print(f"apply(pd.Series): {naive.shape[1]} columns")
print("\nleft behind as dicts:", [c for c in naive.columns if naive[c].map(type).eq(dict).any()])
wide.head(3)

json_normalize: 21 columns
apply(pd.Series): 13 columns

left behind as dicts: ['stopLoss', 'takeProfit', 'extras']


,test,ticker,action,orderType,price,signalPrice,quantity,stopLoss__type,stopLoss__amount,takeProfit__amount,extras__formed_at,extras__score,extras__untouched?,extras__stop_points,extras__total_risk,extras__potential_t1,extras__potential_t2,cancel,ignoreTradingWindows,extras__t2_points,cancelOrderType
0,False,MYM,buy,limit,54103.0,54103.0,16.0,stop,31.0,77.5,2026-08-11 07:30:00-04:00,4.0,3.0,31.0,248.0,186.0,310.0,NaN,NaN,NaN,NaN
1,False,MYM,buy,limit,54103.0,54103.0,16.0,stop,31.0,77.5,2026-08-11 07:30:00-04:00,4.0,3.0,31.0,248.0,186.0,310.0,NaN,NaN,NaN,NaN
2,False,MYM,buy,limit,54103.0,54103.0,16.0,stop,31.0,77.5,2026-08-11 07:30:00-04:00,4.0,3.0,31.0,248.0,186.0,310.0,NaN,NaN,NaN,NaN


In [11]:
# --- Check ---
assert wide.shape == (435, 21), wide.shape
assert "stopLoss__amount" in wide.columns and "extras__score" in wide.columns
assert len(wide) == len(df), "json_normalize must preserve row order and count"
assert naive.shape[1] == 13 < wide.shape[1]
print("✅ Task 5 — 21 flat columns vs 13. The 3 nested objects (stopLoss, takeProfit,")
print("   extras) expanded into 11 leaf columns; apply(pd.Series) left them as dicts.")

✅ Task 5 — 21 flat columns vs 13. The 3 nested objects (stopLoss, takeProfit,
   extras) expanded into 11 leaf columns; apply(pd.Series) left them as dicts.


## Task 6 — Tame the ragged schema

**Why:** `json_normalize` takes the union of all key paths, so rows missing a key get
`NaN`. That's correct and it's also a trap: a column that's 99.8% `NaN` looks like a
broken column and is actually a rare-but-real event type.

**Common approaches:** a sparsity audit — `df.notna().mean().sort_values()` — in one line
tells you which columns are near-empty. Then *decide*, per column, between: drop it,
keep it, or fold it into a categorical "event type" flag. The wrong move is dropping
everything below a threshold without looking.

**Pitfall:** `extras__untouched?` contains a `?`. Column names with punctuation survive
until the moment you try `query()`, `eval()`, `to_parquet()` on a strict engine, or a SQL
export — and then fail somewhere far from the cause. Clean names at the boundary, once.

**Your turn:**
1. Rename every column of `wide` with `to_snake` (it already strips the `extras__` prefix,
   so `extras__potential_t1` → `potential_t1`).
2. Build `sparsity`: fraction non-null per column, ascending.
3. Store the names with <5% coverage in `rare_cols` — and read them before you decide
   anything. (Keep them. They mark the cancel/exit rows.)

In [14]:
wide.columns = [to_snake(c) for c in wide.columns]

sparsity = wide.notna().mean().sort_values()
rare_cols = sparsity[sparsity < 0.05].index.tolist()

print(sparsity.head(8))
print("\nrare:", rare_cols)
# Look before you leap: these 7 rows are position-management events, not corrupt data.
wide.loc[wide[rare_cols].notna().any(axis=1)].dropna(axis=1, how="all")

cancel_order_type         0.002299
ignore_trading_windows    0.004598
cancel                    0.006897
t2_points                 0.016092
untouched                 0.965517
formed_at                 0.967816
score                     0.967816
total_risk                0.967816
dtype: float64

rare: ['cancel_order_type', 'ignore_trading_windows', 'cancel', 't2_points']


,test,ticker,action,order_type,price,signal_price,quantity,stop_loss_type,stop_loss_amount,take_profit_amount,formed_at,score,stop_points,total_risk,potential_t1,potential_t2,cancel,ignore_trading_windows,t2_points,cancel_order_type
421,NaN,MYM,exit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,NaN,NaN
422,False,MYM,buy,limit,52357.0,52357.0,1.0,stop,25.0,52407.0,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN,50.0,NaN
423,True,MYM,buy,limit,52357.0,52357.0,1.0,stop,25.0,52407.0,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN,50.0,NaN
424,True,MYM,buy,limit,52354.0,52354.0,1.0,stop,25.0,52404.0,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN,50.0,NaN
425,False,MYM,breakeven,stop,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,stop
427,False,MYM,buy,limit,47800.0,47800.0,2.0,stop,38.0,380.0,NaN,NaN,38.0,NaN,NaN,NaN,NaN,NaN,380.0,NaN
429,False,MYM,buy,limit,47800.0,47800.0,2.0,stop,38.0,380.0,NaN,NaN,38.0,NaN,NaN,NaN,NaN,NaN,380.0,NaN
430,NaN,MYM,exit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,NaN,NaN
433,False,MYM,buy,limit,47800.0,47800.0,2.0,stop,38.0,380.0,NaN,NaN,38.0,NaN,NaN,NaN,NaN,NaN,380.0,NaN
434,False,MYM,buy,limit,47800.0,47800.0,2.0,stop,38.0,48180.0,2026-03-24 09:30:00,9.0,38.0,38.0,171.0,380.0,NaN,NaN,380.0,NaN


In [13]:
# --- Check ---
assert "untouched" in wide.columns and "?" not in "".join(wide.columns)
assert "potential_t1" in wide.columns, "the extras__ prefix should be gone"
assert "stop_loss_amount" in wide.columns, wide.columns.tolist()
assert set(rare_cols) == {"cancel_order_type", "ignore_trading_windows", "cancel", "t2_points"}, rare_cols
print("✅ Task 6 — the 4 'broken' columns belong to 7 real cancel/exit/breakeven events.")
print("   Threshold-dropping would have deleted your entire position-management history.")

✅ Task 6 — the 4 'broken' columns belong to 7 real cancel/exit/breakeven events.
   Threshold-dropping would have deleted your entire position-management history.


## Task 7 — Right-size the dtypes

**Why:** two reasons, and the second is the important one. Memory: `object` columns store
a pointer per row plus a full Python string; `category` stores a small integer per row plus
one copy of each distinct value. Correctness: a properly typed column *rejects* nonsense.
`Int64` won't hold `"eleven"`; a `category` makes an unexpected new action visible instantly.

**Common approaches:**

- `astype("category")` for low-cardinality strings (`action`, `order_type`, `ticker`) —
  also makes `groupby` faster and gives you an explicit, inspectable value list.
- `astype("Int64")` (capital I) — pandas' **nullable** integer. Plain `int64` cannot hold
  `NaN`, which is why any integer column with a gap silently becomes `float64` and your
  quantities render as `16.0`.
- `astype("boolean")` — nullable boolean, for the 2 rows where `test` is absent. Plain
  `bool` would coerce those `NaN`s to `True`, inventing data.
- `pd.read_csv(..., dtype_backend="pyarrow")` or `convert_dtypes()` to lean on inference —
  handy, but check the result rather than trusting it.

**Pitfall:** `astype(int)` on a column containing `NaN` raises; `astype(float)` "works" and
loses the fact that quantity is a count. Nullable types exist precisely for this.

**Your turn:** produce `typed` from `wide` with: `test` → `boolean`; `ticker`, `action`,
`order_type`, `stop_loss_type`, `cancel_order_type` → `category`; `quantity`, `score`,
`untouched` → `Int64`. Report the memory saving.

In [17]:
# How to change column data types
mem_before = wide.memory_usage(deep=True).sum()

typed = wide.copy()
typed["test"] = typed["test"].astype("boolean")                  # nullable: 2 rows lack the key
for col in ["ticker", "action", "order_type", "stop_loss_type", "cancel_order_type"]:
    typed[col] = typed[col].astype("category")
for col in ["quantity", "score", "untouched"]:
    typed[col] = typed[col].astype("Int64")                      # counts, not measurements

mem_after = typed.memory_usage(deep=True).sum()
print(f"{mem_before/1024:.1f} KB -> {mem_after/1024:.1f} KB "
      f"({100*(1-mem_after/mem_before):.0f}% smaller)")
typed.dtypes

120.9 KB -> 86.2 KB (29% smaller)


test                       boolean
ticker                    category
action                    category
order_type                category
price                      float64
signal_price               float64
quantity                     Int64
stop_loss_type            category
stop_loss_amount           float64
take_profit_amount         float64
formed_at                      str
score                        Int64
untouched                    Int64
stop_points                float64
total_risk                 float64
potential_t1               float64
potential_t2               float64
cancel                      object
ignore_trading_windows      object
t2_points                  float64
cancel_order_type         category
dtype: object

## Task 8 — Rejoin, then derive a feature that surprises you

**Why:** `typed` has the payload contents but no identity — no UUID, no timestamp. Glue
the request metadata back on and you finally have one row per event with everything on it.

**Common approaches:** `pd.concat([left, right], axis=1)` aligns on the **index**, so both
frames must share one — `json_normalize` returns a fresh `RangeIndex`, so call
`.reset_index(drop=True)` on the left before concatenating. (Skip that and you get silent
`NaN`s wherever the indexes disagree, the single most common bug in this whole workflow.)
`df.join()` also aligns on index; `pd.merge()` is for joining on *keys* and is the wrong
tool for a positional stitch.

**Pitfall:** `extras.formed_at` carries an explicit `-04:00` offset. Parse it with
`utc=True` so every value lands on a common timeline regardless of offset; without it,
pandas returns an `object` column of per-row `Timestamp`s the moment offsets differ (which
they will, across a DST boundary), and `.dt` accessors stop working.

**Your turn:**
1. `signals` = metadata (drop `payload`, reset index) concatenated with `typed`.
2. Parse `formed_at` with `utc=True`.
3. Add `age_hours` = hours between `formed_at` and `created_at`.
4. **Look at the distribution before you name it.** Does it match what you'd expect from
   a column measuring "webhook delivery lag"?

In [ ]:
signals = pd.concat(
    [df.drop(columns=["payload"]).reset_index(drop=True), typed],
    axis=1,
)
signals["formed_at"] = pd.to_datetime(signals["formed_at"], utc=True, format="mixed")
signals["age_hours"] = (signals["created_at"] - signals["formed_at"]).dt.total_seconds() / 3600

print(signals.shape)
print(signals["age_hours"].describe())
signals[["created_at_et", "formed_at", "action", "price", "age_hours"]].head()

In [ ]:
# --- Check ---
assert signals.shape[0] == 435 and "uuid" in signals.columns and "price" in signals.columns
assert signals["uuid"].notna().all(), "NaNs in uuid means the concat misaligned — reset_index!"
assert "UTC" in str(signals["formed_at"].dtype), signals["formed_at"].dtype
med = signals["age_hours"].median()
assert 700 < med < 800, med
print(f"✅ Task 8 — median age {med:,.0f} hours ≈ {med/24:.0f} days. Max ≈ {signals['age_hours'].max()/24:.0f} days.")
print("   So 'formed_at' is NOT when the webhook was queued — it's when the price level")
print("   was FORMED, months earlier. The signal fires when price returns to it.")
print("   'delivery_lag' would have been a wrong name on a right number: name derived")
print("   columns after you've looked at their distribution, not before.")

## Task 9 — De-duplicate the replayed webhooks

**Why:** 435 rows is not 435 signals. Compare the first four rows of the raw file: same
ticker, same price, same stop, same everything — four HTTP posts minutes apart. The
strategy re-sends a live setup on every scan cycle until it fills or expires. Analyzing
raw rows would count a single trade idea dozens of times, and every average you compute
would be weighted by how long each setup sat unfilled.

**Common approaches:**

- `df.duplicated()` / `df.drop_duplicates()` over **all** columns — catches nothing here,
  because `uuid` and `created_at` are unique per delivery. That's the lesson: dedupe on
  the columns that define **business identity**, not on the whole row.
- `subset=[...]` + `keep="first"` after **sorting by time** — gives you first-seen. Sorting
  is not optional; `keep="first"` means first *in current row order*.
- `groupby(key).size()` — before you drop anything, count what you're dropping. A key that
  collapses 435 rows to 1 is a broken key.
- `groupby(key)["created_at"].agg(["min", "max", "size"])` — keeps the fan-out as data
  (first fire, last fire, resend count) instead of discarding it.

**Your turn:**
1. Define `SIGNAL_KEY = ["formed_at", "action", "price", "quantity", "test"]` — the fields
   that identify one trade idea.
2. `live` = rows with a non-null `formed_at`, sorted by `created_at`.
3. `resends` = per-key count, descending.
4. `unique_signals` = one row per key, keeping the earliest, plus columns `n_sends` and
   `last_sent_at`.

In [19]:
SIGNAL_KEY = ["formed_at", "action", "price", "quantity", "test"]

live = signals.dropna(subset=["formed_at"]).sort_values("created_at")
resends = live.groupby(SIGNAL_KEY, dropna=False, observed=True).size().sort_values(ascending=False)

# keep="first" on a time-sorted frame == earliest delivery of each idea
unique_signals = live.drop_duplicates(subset=SIGNAL_KEY, keep="first").copy()

# Don't throw the fan-out away — carry it as data.
agg = live.groupby(SIGNAL_KEY, dropna=False, observed=True)["created_at"].agg(["size", "max"])
idx = pd.MultiIndex.from_frame(unique_signals[SIGNAL_KEY])
unique_signals["n_sends"] = agg["size"].reindex(idx).to_numpy()
unique_signals["last_sent_at"] = agg["max"].reindex(idx).to_numpy()

print(f"{len(live)} deliveries -> {len(unique_signals)} distinct signals")
print(resends.head())

421 deliveries -> 35 distinct signals
formed_at                  action  price    quantity  test 
2026-06-17 20:45:00+00:00  sell    52030.0  11        False    88
2026-04-09 11:15:00+00:00  buy     48213.0  15        True     46
2026-05-13 19:15:00+00:00  buy     50118.0  13        True     40
2026-05-14 18:15:00+00:00  sell    50470.0  14        True     32
2026-08-10 20:15:00+00:00  buy     54040.0  16        False    30
dtype: int64


In [20]:
# --- Check ---
assert len(unique_signals) == 35, f"expected 35 distinct signals, got {len(unique_signals)}"
assert unique_signals["n_sends"].sum() == len(live) == 421
assert resends.max() == 88
assert (unique_signals["last_sent_at"] >= unique_signals["created_at"]).all()
print("✅ Task 9 — 421 deliveries collapse to 35 ideas; the busiest was re-sent 88 times.")
print(f"   Raw rows overstate activity by {421/35:.0f}x. Any 'signals per day' chart built")
print("   on the unfiltered file is really a 'how long setups stayed open' chart.")

✅ Task 9 — 421 deliveries collapse to 35 ideas; the busiest was re-sent 88 times.
   Raw rows overstate activity by 12x. Any 'signals per day' chart built
   on the unfiltered file is really a 'how long setups stayed open' chart.


## Task 10 — Drop the test traffic, derive trade metrics

**Why:** 140 of these deliveries carry `"test": true` — dry runs that never hit the broker.
They're indistinguishable from real signals in every other field. Mixing them in inflates
your counts by roughly a third.

**Common approaches:** boolean masks (`df[df["test"] == False]`), `.query("test == False")`
for readability, `np.where` / `np.select` for derived flags, and `.assign()` to chain
several new columns without repeating the frame name.

**Pitfall:** with a **nullable** `boolean` column, `~df["test"]` propagates `NA` for the
2 rows where the flag is missing, and those rows are dropped from *both* `df[mask]` and
`df[~mask]`. Silent disappearance. Be explicit: `== False`, or `.fillna(False)` with the
assumption written down. (Here the 2 missing flags are on cancel events, already excluded
by the `formed_at` filter — but check, don't assume.)

**Your turn:** from `unique_signals`, build `trades` (real signals only) with:
`side` (+1 buy / −1 sell), `rr` = take-profit ÷ stop-loss amount, and
`notional` = price × quantity.

In [25]:
trades = unique_signals[unique_signals["test"] == False].copy()   # explicit: excludes NA
trades = trades.assign(
    side=np.where(trades["action"] == "buy", 1, -1),
    rr=trades["take_profit_amount"] / trades["stop_loss_amount"],
    notional=trades["price"] * trades["quantity"],
)

print(f"{len(unique_signals)} signals -> {len(trades)} real (non-test)")
trades[["session_date", "action", "price", "quantity", "rr", "notional", "score", "n_sends"]].head(10)

35 signals -> 23 real (non-test)


,session_date,action,price,quantity,rr,notional,score,n_sends
434,2026-06-23,buy,47800.0,2,1267.894737,95600.0,9,1
282,2026-07-14,sell,52030.0,11,2.500000,572330.0,9,88
281,2026-07-14,sell,52163.0,10,2.500000,521630.0,7,10
280,2026-07-14,sell,52453.0,10,2.500000,524530.0,7,10
256,2026-07-17,sell,52939.0,7,2.500000,370573.0,8,6
168,2026-07-31,sell,52619.0,12,0.902439,631428.0,4,4
167,2026-07-31,buy,52575.0,8,0.771930,420600.0,4,1
165,2026-07-31,sell,52636.0,9,1.038462,473724.0,4,7
162,2026-07-31,sell,52696.0,9,2.072727,474264.0,4,16
140,2026-08-03,sell,53223.0,11,0.809524,585453.0,6,19


In [26]:
# --- Check ---
assert len(trades) == 23, f"expected 23 real signals, got {len(trades)}"
assert trades["test"].eq(False).all()
assert set(trades["side"].unique()) <= {1, -1}
assert trades["rr"].max() > 1000, "one row has an absurd reward:risk — go find it"
print("✅ Task 10 — 23 real signals. Median R:R", round(trades["rr"].median(), 2))
print("   The max R:R is ~1268 because that row's take_profit_amount is 48,180 on a")
print("   47,800 entry — an absolute PRICE where every other row holds an OFFSET in")
print("   points. A unit bug, not a great trade. This is why you describe() before mean().")

✅ Task 10 — 23 real signals. Median R:R 2.5
   The max R:R is ~1268 because that row's take_profit_amount is 48,180 on a
   47,800 entry — an absolute PRICE where every other row holds an OFFSET in
   points. A unit bug, not a great trade. This is why you describe() before mean().


## Task 11 — Aggregate three ways

**Why:** structured data earns its keep when you can slice it. Three tools cover almost
every question you'll ask.

**Common approaches:**

- `groupby(...).agg(...)` — the workhorse. Named aggregation
  (`agg(n=("uuid", "count"), med=("rr", "median"))`) beats the dict form: explicit output
  names, no MultiIndex columns to flatten afterward.
- `pivot_table(index=, columns=, values=, aggfunc=)` — a `groupby` plus `unstack`, for
  cross-tabs meant to be read by a human. `margins=True` adds totals.
- `resample("D")` on a **DatetimeIndex** — fills in the empty days a `groupby(date)` would
  omit. Essential for time series: a gap is information.

**Pitfall:** `groupby` on a `category` column returns a row for **every** category, even
unobserved combinations, so a small frame explodes into mostly-empty rows. `observed=True`
restricts output to combinations that actually occur. (Pandas is migrating the default;
pass it explicitly so your code behaves the same on either side of the change.)

**Your turn:**
1. `by_action` — per action: count, median R:R, median score, total sends.
2. `by_day` — daily signal counts over the full ET calendar span (zeros included).
3. `test_vs_real` — a pivot of action × test flag.

In [27]:
by_action = (
    trades.groupby("action", observed=True)
          .agg(n=("uuid", "count"),
               median_rr=("rr", "median"),
               median_score=("score", "median"),
               total_sends=("n_sends", "sum"))
          .sort_values("n", ascending=False)
)

by_day = unique_signals.set_index("created_at_et").resample("D").size()

test_vs_real = pd.pivot_table(
    unique_signals, index="action", columns="test",
    values="uuid", aggfunc="count", observed=True, margins=True,
)

display(by_action, by_day[by_day > 0].tail(8), test_vs_real)

,n,median_rr,median_score,total_sends
action,,,,
sell,14,2.5,6.0,211
buy,9,2.5,4.0,73


created_at_et
2026-07-17 00:00:00-04:00    1
2026-07-31 00:00:00-04:00    4
2026-08-03 00:00:00-04:00    4
2026-08-05 00:00:00-04:00    1
2026-08-06 00:00:00-04:00    3
2026-08-07 00:00:00-04:00    1
2026-08-10 00:00:00-04:00    3
2026-08-11 00:00:00-04:00    2
dtype: int64

test,False,True,All
action,,,
buy,9,6,15
sell,14,6,20
All,23,12,35


In [28]:
# --- Check ---
assert {"n", "median_rr"} <= set(by_action.columns), by_action.columns.tolist()
assert by_action["n"].sum() == 23
assert isinstance(by_day.index, pd.DatetimeIndex) and len(by_day) > 35, \
    "resample('D') should span every calendar day, including quiet ones"
assert (by_day == 0).any(), "the zero-signal days are the point"
print("✅ Task 11 —", int((by_day == 0).sum()), "of", len(by_day), "days had no signal at all.")
print("   groupby('session_date') would have hidden every one of them.")

✅ Task 11 — 35 of 50 days had no signal at all.
   groupby('session_date') would have hidden every one of them.


## Task 12 — Wide → long

**Why:** `stop_points`, `total_risk`, `potential_t1`, `potential_t2` are four columns
holding the same *kind* of thing: a points measurement about one signal. In **tidy** form
that's one `metric` column and one `value` column. Plotting libraries (seaborn, plotnine,
Altair) want long form; so does any faceted comparison across metrics.

**Common approaches:** `df.melt(id_vars=, value_vars=, var_name=, value_name=)` is the
standard. `pd.wide_to_long` handles the `metric_2020`, `metric_2021` naming pattern.
`.stack()` melts *everything* not in the index. Round-trip back with `pivot`/`pivot_table`.

**Pitfall:** melting a frame with mixed dtypes collapses them into one `object` column —
you can't melt an int column and a string column together and keep both types. Melt
homogeneous groups only, and cast the result if you need it numeric.

**Your turn:** melt `trades` into `long` keyed by `uuid`, `session_date`, `action`, over the
four metrics; drop null values; then pivot it back and confirm the round-trip.

In [29]:
METRICS = ["stop_points", "total_risk", "potential_t1", "potential_t2"]

long = trades.melt(
    id_vars=["uuid", "session_date", "action"],
    value_vars=METRICS,
    var_name="metric",
    value_name="value",
).dropna(subset=["value"])

roundtrip = long.pivot(index="uuid", columns="metric", values="value")

print(long.shape)
display(long.head(6), long.groupby("metric", observed=True)["value"].describe()[["count", "50%", "max"]])

(92, 5)


,uuid,session_date,action,metric,value
0,2393f062-2ba0-428e-a587-c5492cded526,2026-06-23,buy,stop_points,38.0
1,e9239174-b07d-4551-a63b-ecb252352377,2026-07-14,sell,stop_points,45.0
2,8e4f7e52-0069-4345-8e2f-80f20a5192e5,2026-07-14,sell,stop_points,46.0
3,36f361ad-8efd-49c7-ba38-a6c72b7c7192,2026-07-14,sell,stop_points,46.0
4,a9295fb8-d3fd-4812-ac88-29c5ab6d8845,2026-07-17,sell,stop_points,64.0
5,6357feb9-6d28-4e08-8901-070585351a4d,2026-07-31,sell,stop_points,41.0


,count,50%,max
metric,,,
potential_t1,23.0,176.0,255.0
potential_t2,23.0,300.0,380.0
stop_points,23.0,46.0,87.0
total_risk,23.0,240.0,248.5


In [30]:
# --- Check ---
assert set(long.columns) == {"uuid", "session_date", "action", "metric", "value"}
assert long["metric"].nunique() == 4
assert len(long) == long["value"].notna().sum()
assert roundtrip.shape[0] == trades["uuid"].nunique()
print(f"✅ Task 12 — {len(trades)} rows x 4 metrics -> {len(long)} long rows, round-trip clean.")

✅ Task 12 — 23 rows x 4 metrics -> 92 long rows, round-trip clean.


## Task 13 — Validate, then persist

**Why:** the cleaning you just did is a set of *assumptions* — one ticker, four action
types, positive quantities, unique IDs. Written as a function, those assumptions become a
regression test that runs against next month's export and tells you the day the strategy
starts emitting something new.

**Common approaches:** a plain `check(df) -> list[str]` function costs nothing and catches
most of it. `pandera` gives you declarative schemas with coercion and rich errors;
`pydantic` validates *before* the DataFrame exists, at the JSON boundary — the better place
for a webhook receiver; `great-expectations` is the heavyweight for scheduled pipelines.

**On persistence:** CSV round-trips lose every dtype you just set — your `category` comes
back as `object`, your tz-aware timestamps as strings, your `Int64` as `float64`. Parquet
stores the schema with the data, compresses well, and reads back identical. Use Parquet
for analysis artifacts, JSONL when something downstream expects one record per line, and
SQLite when you want to query without loading.

**Pitfall:** `to_json` can't serialize tz-aware timestamps or `pd.NA` without help — cast
them to ISO strings first. Parquet handles both natively, which is most of the argument.

**Your turn:**
1. Write `check(frame)` returning a list of problem strings (empty = clean). Cover:
   duplicate `uuid`, unexpected `action`, non-positive `quantity`, null `created_at`,
   more than one `ticker`.
2. Run it on `trades`.
3. Write `signals_clean.parquet`, `signals_clean.jsonl`, and a `signals` table in
   `signals.db`; compare the file sizes.

In [31]:
VALID_ACTIONS = {"buy", "sell", "exit", "breakeven"}

def check(frame: pd.DataFrame) -> list[str]:
    problems = []
    if frame["uuid"].duplicated().any():
        problems.append("duplicate uuid")
    unexpected = set(frame["action"].dropna().unique()) - VALID_ACTIONS
    if unexpected:
        problems.append(f"unexpected action(s): {unexpected}")
    if (frame["quantity"].dropna() <= 0).any():
        problems.append("non-positive quantity")
    if frame["created_at"].isna().any():
        problems.append("null created_at")
    if frame["ticker"].nunique() > 1:
        problems.append("multiple tickers — check your grouping assumptions")
    return problems

print("problems:", check(trades))

OUT = Path(".")

# Parquet keeps dtypes, tz-awareness and categories intact.
trades.to_parquet(OUT / "signals_clean.parquet", index=False)

# JSON has no timestamp type: stringify the datetimes on the way out.
(trades.assign(**{c: trades[c].astype(str)
                  for c in ["created_at", "created_at_et", "formed_at", "last_sent_at", "session_date"]})
       .to_json(OUT / "signals_clean.jsonl", orient="records", lines=True))

# SQLite likewise: naive-ise/stringify anything it can't store.
with sqlite3.connect(OUT / "signals.db") as con:
    (trades.assign(**{c: trades[c].astype(str)
                      for c in ["created_at", "created_at_et", "formed_at", "last_sent_at", "session_date"]})
           .to_sql("signals", con, if_exists="replace", index=False))

for f in ["signals_clean.parquet", "signals_clean.jsonl", "signals.db"]:
    print(f"{f:<24} {(OUT / f).stat().st_size/1024:>7.1f} KB")

problems: []
signals_clean.parquet       22.6 KB
signals_clean.jsonl         17.0 KB
signals.db                  16.0 KB


## ★ Bonus — pull structure out of the strings you discarded

Two columns you dropped in Task 2 still have information in them, reachable with the `.str`
accessor (vectorized string ops) rather than a Python loop.

**Common approaches:** `.str.extract(pattern)` returns capture groups as columns —
the standard tool when a field packs several values into one string. `.str.split(..., expand=True)`
when the format is delimited and stable. `.str.contains` for filtering. And when a value has a
real type in the standard library, use it: `ipaddress.ip_address()` parses and classifies
correctly, where a naive `":" in s` test is a guess that works until it doesn't.

**Your turn:**
1. Extract the webhook endpoint UUID from `raw["URL"]` (the 36-char UUID after `/webhook/`)
   into `webhook_id`.
2. Classify each `raw["IP Address"]` as IPv4 or IPv6 into `ip_version`.
3. Cross-tabulate IP version against ET hour of day — the IPv6 traffic is a person on a
   phone or laptop, the IPv4 block is a datacenter.

In [32]:
import ipaddress

webhook_id = raw["URL"].str.extract(r"/webhook/([0-9a-f-]{36})/")[0]
ip_version = raw["IP Address"].map(lambda s: f"IPv{ipaddress.ip_address(s).version}")

by_hour = pd.crosstab(df["created_at_et"].dt.hour, ip_version)
by_hour.index.name = "hour_et"
display(by_hour)

IP Address,IPv4,IPv6
hour_et,,
1,0,3
3,0,1
6,0,4
8,76,0
9,65,0
10,50,0
11,70,0
12,71,0
13,36,0


In [33]:
# --- Check ---
assert webhook_id.nunique() == 1 and webhook_id.iloc[0].count("-") == 4
assert ip_version.value_counts().to_dict() == {"IPv4": 421, "IPv6": 14}
print("✅ Bonus — endpoint", webhook_id.iloc[0])
print("   421 IPv4 deliveries (mostly one 74.220.49.x block — a hosted scheduler) and")
print("   14 from IPv6 addresses on a different network. Worth a source flag in a")
print("   real pipeline: automated and hand-fired signals deserve separate treatment.")

✅ Bonus — endpoint 5b4d401a-aea8-4e8e-9014-dad57406cd08
   421 IPv4 deliveries (mostly one 74.220.49.x block — a hosted scheduler) and
   14 from IPv6 addresses on a different network. Worth a source flag in a
   real pipeline: automated and hand-fired signals deserve separate treatment.


## Where to go next

**Same job, different engine.** The pandas idioms above are the ones you'll meet in most
codebases, but they aren't the only option:

- **Polars** — `pl.read_csv(...).with_columns(pl.col("payload").str.json_decode())` decodes
  JSON into a native struct column you can `.unnest()`. Multi-threaded, lazy, and typically
  several times faster on files large enough to notice.
- **DuckDB** — `SELECT payload->>'$.ticker' FROM read_csv_auto('file.csv')` queries the CSV
  in place with JSON path operators, no load step. Ideal when the file is bigger than RAM
  or you'd rather think in SQL.
- **pandas + PyArrow backend** — `read_csv(..., dtype_backend="pyarrow")` gets you proper
  nullable types and better string performance without leaving the pandas API.

**Harder variants of this dataset**, in rough order of difficulty:

1. Payloads containing a **list** of legs — that's `json_normalize(record_path=..., meta=...)`,
   the part of the API this file didn't exercise.
2. Reconstruct **position lifecycles**: match each `buy`/`sell` entry to its later `exit` or
   `breakeven` for the same setup, and compute how long each was live.
3. Turn the whole notebook into an idempotent `clean(path) -> DataFrame` function with the
   Task 13 checks as a pytest suite — then run it against a fresh export and see what breaks.
4. Model the resend behaviour: does the number of resends before an entry correlate with the
   strategy `score`? (Careful — that's an observational question with obvious confounders.)

**One habit worth keeping.** Every non-obvious finding here came from looking at the data
rather than from a transform: the 421→35 collapse, `formed_at` meaning something other than
its name suggested, the R:R of 1268, the 14 manual re-fires. The transforms are mechanical
and you'll memorize them in a week. Checking whether the numbers you produced *could
plausibly be true* is the part that stays hard, and it's the reason to work cell by cell.